In [1]:
#import libraries
import os
import sys
import numpy as np
import pandas as pd


In [2]:
climate_plan = pd.read_csv(os.path.join('..','..','data','raw','lgcap_survey_raw_data.csv'))


In [3]:
climate_plan.head()

,Year,Org Unit,MUNI,RD,EDR,1: FTEs,2: Climate Action Plan,"2: addressing mitigation, adaptation and/or energy use",2: addressing mitigation and adaptation,2: addressing mitigation and energy,...,20: LGCAP funding for year one- Reserve funding (2),20: LGCAP funding for year one-Reserve funding - No decision has been made.,21: Additional funding leveraged,22: total mitigation investment,23: expected co2e reductions,24: total investment in adaptation and resilience,25: use a climate lens on infrastructure planning and decision-making?,2020 pop,Size,GEO
0,2022,5941005,100 Mile House,Cariboo Regional District,Cariboo,0.25,Yes,NaN,NaN,NaN,...,NaN,No decision has been made.,0.0,0.0,NaN,0.00,No,1881,Small,"100 Mile House, British Columbia, Canada"
1,2022,5909052,Abbotsford,Fraser Valley Regional District,Mainland/Southwest,5.50,No,NaN,NaN,NaN,...,NaN,No decision has been made.,9100000.0,1173563.0,0.0,3350077.63,No,161581,Large,"Abbotsford, British Columbia, Canada"
2,2022,1005923,Alberni-Clayoquot Regional District,Alberni-Clayoquot Regional District,Vancouver Island/Coast,0.50,No,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,NaN,0.00,No,10362,Small,"Alberni-Clayoquot Regional District, British C..."
3,2022,5943008,Alert Bay,Mount Waddington Regional District,Vancouver Island/Coast,0.10,No,NaN,NaN,NaN,...,NaN,No decision has been made.,0.0,0.0,NaN,0.00,No,441,Small,"Alert Bay, British Columbia, Canada"
4,2022,5915038,Anmore,Metro Vancouver Regional District,Mainland/Southwest,0.00,No,NaN,NaN,NaN,...,NaN,NaN,0.0,0.0,0.0,0.00,No,2412,Small,"Anmore, British Columbia, Canada"


In [4]:
# Create a mapping dictionary for community name standardization
community_name_mapping = {
    'Langley, City of': 'Langley City',
    'Langley, District Municipality': 'Langley Township',
    'North Vancouver, City of': 'North Vancouver City',
    'North Vancouver, District Municipality': 'North Vancouver District',
    '100 Mile House': 'One Hundred Mile House',
    'Daajing Giids': 'Queen Charlotte',
    'Sechelt District Municipality': 'Sechelt'
}

# Apply the mapping to the MUNI column
climate_plan['MUNI'] = climate_plan['MUNI'].replace(community_name_mapping)

In [5]:
# Get community names used in the wellbeing dataset to match with remoteness dataset
wellbeing_communities = pd.read_csv(os.path.join('..','..','data','processed','matched_cwb_2021.csv'), usecols=['CSD Name 2021'])

In [6]:
# Get unique community names from each dataset as a list
climate_plan_communities = set(climate_plan['MUNI'].dropna())

# Find exact matches (as a list)
exact_matches_2021 = climate_plan_communities.intersection(set(wellbeing_communities['CSD Name 2021']))

print(f"Exact matches with 2021 wellbeing: {len(exact_matches_2021)}")

# Get unmatched communities by subtracting lists
unmatched_communities = set(wellbeing_communities['CSD Name 2021']) - exact_matches_2021

print("\nAll unmatched communities:")
for community in sorted(list(unmatched_communities)):
    print(f"  {community}")

Exact matches with 2021 wellbeing: 147

All unmatched communities:


In [7]:
climate_plan_subset = climate_plan[climate_plan['MUNI'].isin(exact_matches_2021)].copy()

In [8]:
print(climate_plan_subset.columns.tolist())

['Year', 'Org Unit', 'MUNI', 'RD', 'EDR', '1: FTEs', '2:\xa0Climate Action Plan', '2: addressing mitigation, adaptation and/or energy use', '2: addressing mitigation and adaptation', '2: addressing mitigation and energy', '2: addressing adaptation and energy', '2: Standalone mitigation plan', '2: Standalone adaptation plan', '2: Standalone energy-related plan', "2: Don't know", 'Please include a link to the document or webpage if available.', '2: No, but we are currently undertaking one and it will be completed in the next two years.', '2: No, we are not intending to undertake one due to lack of financial capacity.', '2: No, we are not intending to undertake one due to lack of expertise or technical capacity.', '3: climate emergency', '4: Challenges.Lack of jurisdiction.', '4: Challenges.Lack of staff capacity or expertise.', '4: Challenges.Lack of financial resources.', '4: Challenges.Lack of data or information.', '4: Challenges.Lack of local political support.', '4: Challenges.Lack 

In [9]:
climate_plan_subset.columns = climate_plan_subset.columns.str.replace(r'\s+', ' ', regex=True)
# Then use the normal name:
climate_plan_subset['Has_Climate_Action_Plan'] = climate_plan_subset['2: Climate Action Plan'].map({'Yes': 1, 'No': 0})
climate_plan_subset['Climate_Emergency'] = climate_plan_subset['3: climate emergency'].map({'Yes': 1, 'No': 0})
climate_plan_subset['Reports_Corporate_Emissions'] = climate_plan_subset['5: reported corporate GHG emissions'].map({'Yes': 1, 'No': 0, 'No, but for a past year. (Please enter most recent year completed: YYYY)': 1})
#if either 6: measured and reported community-wide emissions in the past? or '5: reported corporate GHG emissions' is yes, then set Community_GHG_tracking to 1
climate_plan_subset['Community_GHG_tracking'] = climate_plan_subset['6: community-wide GHG emissions'].map({'Yes': 1, 'No': 0, 'In-progress': 1})
climate_plan_subset.loc[climate_plan_subset['6: measured and reported community-wide emissions in the past?'] == 'Yes', 'Community_GHG_tracking'] = 1
climate_plan_subset['Net_Zero_Or_Carbon_Neutral_Target'] = climate_plan_subset['8: net-zero or carbon-neutral emissions target?'].map(
    {'Yes: Net-zero (Net-zero refers to a jurisdiction achieving a balance between greenhouse gas emissions produced and removed from the atmosphere (e.g. planting trees or using carbon capture technologies))': 1, 
     'Yes: Carbon neutral (Carbon neutral refers to a jurisdiction achieving a state of no net carbon dioxide emissions being released into the atmosphere (e.g. Reducing emissions as far as practical and purchasing offsets or other similar mechanisms))': 1,
     'No': 0})



In [10]:
climate_plan_subset['No_Staff_Capacity_For_GHG_Tracking'] = climate_plan_subset[
    "6: No, we do not measure and report community-wide emissions data due to lack of staff and technical capacity."
    ].map({"No, we do not measure and report community-wide emissions data due to lack of staff and technical capacity.": 1, np.nan: 0})

climate_plan_subset['No_Financial_Capacity_For_GHG_Tracking'] = climate_plan_subset[
    "6: No, we do not measure and report community-wide emissions data due to lack of financial capacity."
    ].map({"No, we do not measure and report community-wide emissions data due to lack of financial capacity.": 1, np.nan: 0})

climate_plan_subset['Lack_Of_Jursidiction_Climate_Action'] = climate_plan_subset[
    "4: Challenges.Lack of jurisdiction."].map({"Lack of jurisdiction.": 1, np.nan: 0})

climate_plan_subset['Lack_Of_Staff_Capacity_Climate_Action'] = climate_plan_subset[
    "4: Challenges.Lack of staff capacity or expertise."].map({"Lack of staff capacity or expertise.": 1, np.nan: 0})

climate_plan_subset['Lack_Of_Financial_Capacity_Climate_Action'] = climate_plan_subset[
    "4: Challenges.Lack of financial resources."].map({"Lack of financial resources.": 1, np.nan: 0})

climate_plan_subset['Lack_Of_Information_Climate_Action'] = climate_plan_subset[
    "4: Challenges.Lack of data or information."].map({"Lack of data or information.": 1, np.nan: 0})

climate_plan_subset['Lack_Of_Political_Support_Climate_Action'] = climate_plan_subset[
    "4: Challenges.Lack of local political support."].map({"Lack of local political support.": 1, np.nan: 0})

climate_plan_subset['Lack_Of_Provincial_Federal_Support_Climate_Action'] = climate_plan_subset[
    "4: Challenges.Lack of provincial or federal government support or collaboration."].map({"Lack of provincial or federal government support or collaboration.": 1, np.nan: 0})

In [11]:
climate_plan_subset.to_csv(os.path.join('..','..','data','processed','climate_action_plans_matched.csv'), index=False)

In [12]:
climate_plan_subset.head()

,Year,Org Unit,MUNI,RD,EDR,1: FTEs,2: Climate Action Plan,"2: addressing mitigation, adaptation and/or energy use",2: addressing mitigation and adaptation,2: addressing mitigation and energy,...,Community_GHG_tracking,Net_Zero_Or_Carbon_Neutral_Target,No_Staff_Capacity_For_GHG_Tracking,No_Financial_Capacity_For_GHG_Tracking,Lack_Of_Jursidiction_Climate_Action,Lack_Of_Staff_Capacity_Climate_Action,Lack_Of_Financial_Capacity_Climate_Action,Lack_Of_Information_Climate_Action,Lack_Of_Political_Support_Climate_Action,Lack_Of_Provincial_Federal_Support_Climate_Action
0,2022,5941005,One Hundred Mile House,Cariboo Regional District,Cariboo,0.25,Yes,NaN,NaN,NaN,...,1,0,0,0,1,1,0,1,0,0
1,2022,5909052,Abbotsford,Fraser Valley Regional District,Mainland/Southwest,5.50,No,NaN,NaN,NaN,...,1,0,0,0,0,1,1,0,0,1
3,2022,5943008,Alert Bay,Mount Waddington Regional District,Vancouver Island/Coast,0.10,No,NaN,NaN,NaN,...,0,0,1,0,0,1,1,1,0,0
4,2022,5915038,Anmore,Metro Vancouver Regional District,Mainland/Southwest,0.00,No,NaN,NaN,NaN,...,0,0,1,1,0,1,1,0,0,0
5,2022,5937028,Armstrong,North Okanagan Regional District,Thompson/Okanagan,0.00,No,NaN,NaN,NaN,...,0,0,1,0,0,1,1,1,0,0


In [13]:
#print all column names 
print(climate_plan_subset.columns.tolist())

['Year', 'Org Unit', 'MUNI', 'RD', 'EDR', '1: FTEs', '2: Climate Action Plan', '2: addressing mitigation, adaptation and/or energy use', '2: addressing mitigation and adaptation', '2: addressing mitigation and energy', '2: addressing adaptation and energy', '2: Standalone mitigation plan', '2: Standalone adaptation plan', '2: Standalone energy-related plan', "2: Don't know", 'Please include a link to the document or webpage if available.', '2: No, but we are currently undertaking one and it will be completed in the next two years.', '2: No, we are not intending to undertake one due to lack of financial capacity.', '2: No, we are not intending to undertake one due to lack of expertise or technical capacity.', '3: climate emergency', '4: Challenges.Lack of jurisdiction.', '4: Challenges.Lack of staff capacity or expertise.', '4: Challenges.Lack of financial resources.', '4: Challenges.Lack of data or information.', '4: Challenges.Lack of local political support.', '4: Challenges.Lack of 

In [ ]:
import pandas as pd


# Instead of 13 binary variables:
climate_plan_subset['Climate_Action_Score'] = (
    climate_plan_subset['Has_Climate_Action_Plan'] + 
    climate_plan_subset['Climate_Emergency'] + 
    climate_plan_subset['Reports_Corporate_Emissions'] +
    climate_plan_subset['Net_Zero_Or_Carbon_Neutral_Target'] +
    climate_plan_subset['Community_GHG_tracking']
)  # 0-5 scale

climate_plan_subset['Climate_Action_N_Barriers'] = (
    climate_plan_subset['No_Staff_Capacity_For_GHG_Tracking'] +
    climate_plan_subset['No_Financial_Capacity_For_GHG_Tracking'] +
    climate_plan_subset['Lack_Of_Staff_Capacity_Climate_Action'] +
    climate_plan_subset['Lack_Of_Financial_Capacity_Climate_Action'] +
    climate_plan_subset['Lack_Of_Information_Climate_Action'] +
    climate_plan_subset['Lack_Of_Political_Support_Climate_Action'] +
    climate_plan_subset['Lack_Of_Provincial_Federal_Support_Climate_Action'] +
    climate_plan_subset['Lack_Of_Jursidiction_Climate_Action']
)  # Count of barriers


# List all your binary/metric columns (the ones you want to convert to tidy)
columns = [
    'Has_Climate_Action_Plan',
    'Climate_Emergency',
    'Reports_Corporate_Emissions',
    'Community_GHG_tracking',
    'Net_Zero_Or_Carbon_Neutral_Target',
    'No_Staff_Capacity_For_GHG_Tracking',
    'No_Financial_Capacity_For_GHG_Tracking',
    'Lack_Of_Jursidiction_Climate_Action',
    'Lack_Of_Staff_Capacity_Climate_Action',
    'Lack_Of_Financial_Capacity_Climate_Action',
    'Lack_Of_Information_Climate_Action',
    'Lack_Of_Political_Support_Climate_Action',
    'Lack_Of_Provincial_Federal_Support_Climate_Action',
    'Climate_Action_N_Barriers',
    'Climate_Action_Score'
]

# Melt: convert wide to tidy (one row per municipality × metric)
climate_plan_tidy = climate_plan_subset[['MUNI'] + columns].melt(
    id_vars=['MUNI'],
    value_vars=columns,
    var_name='Metric',
    value_name='Value'
).copy()



# Rename and add columns to match tidy format
climate_plan_tidy = climate_plan_tidy.rename(columns={'MUNI': 'Municipality_Name'})
climate_plan_tidy['Year'] = 2021
climate_plan_tidy['Unit'] = 'Yes/No'

# Update units for summary scores
climate_plan_tidy.loc[climate_plan_tidy['Metric'] == 'Climate_Action_Score', 'Unit'] = 'count out of 5'
climate_plan_tidy.loc[climate_plan_tidy['Metric'] == 'Climate_Action_N_Barriers', 'Unit'] = 'count out of 8'


# Reorder to standard tidy column order
climate_plan_tidy = climate_plan_tidy[['Municipality_Name', 'Year', 'Metric', 'Value', 'Unit']]

# Save to CSV
climate_plan_tidy.to_csv(os.path.join('..','..','data','processed','tidy_climate_action.csv'), index=False)

print(f"Created {len(climate_plan_tidy)} tidy rows ({len(climate_plan_subset)} municipalities × {len(columns)} metrics)")
print("\nFirst few rows:")
display(climate_plan_tidy.tail(10))

Created 2205 tidy rows (147 municipalities × 15 metrics)

First few rows:


,Municipality_Name,Year,Metric,Value,Unit
2195,Vancouver,2021,Climate_Action_Score,5,count out of 5
2196,Vanderhoof,2021,Climate_Action_Score,0,count out of 5
2197,Vernon,2021,Climate_Action_Score,4,count out of 5
2198,Victoria,2021,Climate_Action_Score,4,count out of 5
2199,View Royal,2021,Climate_Action_Score,5,count out of 5
2200,Warfield,2021,Climate_Action_Score,1,count out of 5
2201,West Vancouver,2021,Climate_Action_Score,5,count out of 5
2202,Whistler,2021,Climate_Action_Score,4,count out of 5
2203,White Rock,2021,Climate_Action_Score,2,count out of 5
2204,Williams Lake,2021,Climate_Action_Score,0,count out of 5
